# Lecture 15 — Testing, documentation and continuous integration

A test runs the code on an input whose correct result is known, and checks the result.

Notes: [Lecture 15](https://teaching.borisbolliet.io/c1-2026/materials/class15/notes.pdf) ·
code: [`attractor` at `lecture-15-start`](https://github.com/borisbolliet/attractor/tree/lecture-15-start/)

The notebook runs at the root of the `attractor` clone, at the tag `lecture-15-start`, with the kernel
*Python (attractor 3.13)*. The lock now also holds pytest and pytest-cov:

```bash
cd ~/GitHub/attractor
git fetch --tags --force
git switch --detach lecture-15-start
source setup_env.sh
```

pytest, coverage and Zensical run as commands, through the helper `sh`. The files they read are shown
in the Markdown cell before each command. The cells that change the code or the tests work in a
temporary clone, `work`. They never change the clone of the notebook.

In [ ]:
import importlib.util, math, os, shlex, shutil, subprocess, sys, tempfile
from pathlib import Path

import numpy as np
import pytest

TAG = "lecture-15-start"
REPO = "https://github.com/borisbolliet/attractor.git"
if not Path("src/attractor").exists():   # outside a clone (Colab, a download): fetch and install the project
    root = Path(tempfile.mkdtemp()) / "attractor"
    subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "-q", "--branch", TAG, REPO, str(root)],
                   check=True)
    os.chdir(root)
    if importlib.util.find_spec("attractor") is None:   # e.g. Colab: install the project into this kernel
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[test,fit]"], check=True)
        sys.path.insert(0, str(root / "src"))
BIN = Path(sys.executable).parent        # the bin/ folder of the kernel's environment

def sh(cmd, cwd=None, tail=None, src=None):
    """Run a shell command, print its output (the last `tail` lines) and its exit status if not 0.
    With src, the package is imported from that folder instead of the installed one."""
    env = {**os.environ, "PIP_DISABLE_PIP_VERSION_CHECK": "1", "PYTHON_COLORS": "0", "NO_COLOR": "1"}
    env["PATH"] = f"{BIN}{os.pathsep}{env['PATH']}"
    if src:   # the temporary clone: its own src/, and no .pyc that could outlive an edit of the same size
        env["PYTHONPATH"] = str(src)
        env["PYTHONDONTWRITEBYTECODE"] = "1"
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, env=env)
    lines = (r.stdout + r.stderr).rstrip().splitlines()
    print("\n".join(lines[-tail:] if tail else lines))
    if r.returncode:
        print(f"(exit status {r.returncode})")

def py(source, cwd=None, src=None):
    """Run Python source in a new interpreter, as sh does for a command."""
    sh(f"{sys.executable} -c {shlex.quote(source)}", cwd=cwd, src=src)

def edit(path, old, new):
    """Replace the line old by new in the file path (exactly one occurrence)."""
    text = path.read_text()
    assert text.count(old) == 1, old
    path.write_text(text.replace(old, new))

subprocess.run(["git", "describe", "--tags"], capture_output=True, text=True).stdout.strip()

## 1 Tests with pytest

### A first test

A **test** is a function that calls the code and checks the result with `assert`. A false `assert`
raises `AssertionError`, and the test fails.

pytest collects the files `test_*.py` and, in them, the functions `test_*`. The three tests of
[`tests/test_integrators.py`](https://github.com/borisbolliet/attractor/blob/lecture-15-start/tests/test_integrators.py):

```python
"""Tests of the integrators, before Lecture 15."""
import math

from attractor.integrators import euler_step, integrate, rk4_step


def test_constant_rhs():
    assert euler_step(lambda t, y: 2.0, 0.0, 1.0, 0.5) == 2.0
    assert rk4_step(lambda t, y: 2.0, 0.0, 1.0, 0.5) == 2.0


def test_shapes():
    t, y = integrate(rk4_step, lambda t, y: -y, [1.0, 2.0], 0.0, 1.0, 0.1)
    assert t.shape == (11,)
    assert y.shape == (11, 2)


def test_decay():
    t, y = integrate(rk4_step, lambda t, y: -y, [1.0], 0.0, 1.0, 0.1)
    assert math.isclose(y[-1, 0], math.exp(-1), rel_tol=1e-3)
```

`python -m pytest` runs them with the kernel's interpreter. The option `-v` prints one line per test.

In [ ]:
sh(f"{sys.executable} -m pytest -v -p no:cacheprovider", tail=5)

### Coverage

The **coverage** of a module is the fraction of its lines that the tests execute. `--cov-branch` also
counts the two outcomes of every `if`. `Missing` lists the lines that no test ran.

In [ ]:
sh(f"{sys.executable} -m pytest -q -p no:cacheprovider --cov=attractor --cov-branch "
   "--cov-report=term-missing", tail=16)

The lines missed in `integrators.py` are the methods of `Euler` and `RK4`. Every line of `rk4_step` ran.

### Every line runs, and the code is wrong

`rk4_step` differs from the one of `lecture-12` in one character: `k3` is built from `k1`, not `k2`.

In [ ]:
sh("git diff lecture-12 lecture-15-start -- src/attractor/integrators.py")

At $h = 0.1$, the wrong RK4 gives $y(1)$ for $dy/dt = -y$ with a relative error of $8.8 \times 10^{-4}$.
That is below the tolerance $10^{-3}$ of `test_decay`, so the test passes.

In [ ]:
from attractor.integrators import euler_step, integrate, rk4_step

def rk4_correct(rhs, t, y, h):
    """The classical RK4 step, with k3 from k2."""
    k1 = rhs(t, y)
    k2 = rhs(t + h / 2, y + h / 2 * k1)
    k3 = rhs(t + h / 2, y + h / 2 * k2)
    k4 = rhs(t + h, y + h * k3)
    return y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

decay = (lambda t, y: -y), math.exp(-1)

def error_at_1(step, problem, h):
    rhs, exact = problem
    t, y = integrate(step, rhs, [1.0], 0.0, 1.0, h)
    return abs(y[-1, 0] - exact)

for name, step in [("rk4 (k3 from k2)", rk4_correct), ("rk4 of lecture-15-start", rk4_step),
                   ("euler", euler_step)]:
    print(f"{name:24s} relative error {error_at_1(step, decay, 0.1) / math.exp(-1):.2g}")

## 2 Testing a property: the order of convergence

A method of **order** $p$ has an error $e(h) \approx C h^p$. Halving the step divides the error by $2^p$,
so $p \approx \log_2 \big(e(h) / e(h/2)\big)$.

In [ ]:
for name, step in [("euler", euler_step), ("rk4 (k3 from k2)", rk4_correct),
                   ("rk4 of lecture-15-start", rk4_step)]:
    ratio = error_at_1(step, decay, 0.1) / error_at_1(step, decay, 0.05)
    print(f"{name:24s} ratio {ratio:5.1f}   order {math.log2(ratio):.2f}")

The tests of `lecture-15` check this order. In [`tests/test_integrators.py`](https://github.com/borisbolliet/attractor/blob/lecture-15/tests/test_integrators.py):

```python
def error_at_1(step, problem, h):
    """Absolute error at t = 1 of a fixed-step run of size h."""
    rhs, exact = problem
    t, y = integrate(step, rhs, [1.0], 0.0, 1.0, h)
    return abs(y[-1, 0] - exact)


@pytest.mark.parametrize("step, order", [(euler_step, 1), (rk4_step, 4)])
def test_convergence_order(step, order, problem):
    """Halving h divides the error by about 2**order."""
    ratio = error_at_1(step, problem, 0.1) / error_at_1(step, problem, 0.05)
    assert math.log2(ratio) == pytest.approx(order, abs=0.2)
```

We make a temporary clone `work` of `lecture-15-start` and copy the tests of `lecture-15` into it.

In [ ]:
work = Path(tempfile.mkdtemp()) / "work"
subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "-q", ".", str(work)], check=True)
subprocess.run(["git", "-c", "advice.detachedHead=false", "switch", "-q", "--detach", "lecture-15-start"],
               cwd=work, check=True)
sh("git checkout lecture-15 -- tests pyproject.toml", cwd=work)

We run them on the code of `lecture-15-start`. `src=` makes the clone's own `src/` the package imported.

In [ ]:
sh(f"{sys.executable} -m pytest -q -p no:cacheprovider", cwd=work, src=work / "src", tail=14)

The measured order of the RK4 is 2.04, not 4. Seven of the failures are tests of exceptions that the
code of `lecture-15-start` does not raise yet (Section 4). With the code of `lecture-15` too, every test
passes:

In [ ]:
sh("git checkout lecture-15 -- src docs zensical.toml .github", cwd=work)
sh(f"{sys.executable} -m pytest -q -p no:cacheprovider", cwd=work, src=work / "src", tail=1)

## 3 Floating-point comparison

A computed float rarely equals a decimal value. A test compares within a **tolerance**: relative,
$|a - b| \le r\,|b|$, or absolute, $|a - b| \le \varepsilon$.

In [ ]:
0.1 + 0.2 == 0.3, 0.1 + 0.2

In [ ]:
math.isclose(0.1 + 0.2, 0.3), 0.1 + 0.2 == pytest.approx(0.3)

Near zero a relative tolerance accepts almost nothing, and an absolute one can accept anything.
Forward Euler gives $y(10)$ for $dy/dt = -y$ with an error of 41 %:

In [ ]:
t, y = integrate(euler_step, lambda t, y: -y, [1.0], 0.0, 10.0, 0.1)
y10 = y[-1, 0]
print(y10, np.exp(-10))
y10 == pytest.approx(np.exp(-10)), y10 == pytest.approx(np.exp(-10), abs=1e-4)

## 4 Fixtures, parametrisation and exceptions

A **fixture** prepares the input of a test. A test asks for it by naming it as an argument. The fixture
`problem` of [`tests/conftest.py`](https://github.com/borisbolliet/attractor/blob/lecture-15/tests/conftest.py) has two values, so every test that uses it runs twice:

```python
@pytest.fixture(params=["decay", "forced"])
def problem(request):
    """An equation dy/dt = rhs(t, y) with y(0) = 1, and its exact solution at t = 1.

    The fixture has two values, so every test that uses it runs twice.
    "forced" depends on t explicitly; "decay" does not.
    """
    if request.param == "decay":
        return (lambda t, y: -y), np.exp(-1.0)
    return (lambda t, y: np.cos(t) * y), np.exp(np.sin(1.0))
```

`@pytest.mark.parametrize` runs a test once per value. Two decorators give every combination, here
2 × 3 = 6 runs:

```python
@pytest.mark.parametrize("method", [Euler, RK4])
@pytest.mark.parametrize("h", [0.0, -0.1, float("nan")])
def test_step_must_be_positive(method, h):
    with pytest.raises(ValueError, match="must be positive"):
        method(h)
```

In [ ]:
sh(f"{sys.executable} -m pytest -v -p no:cacheprovider tests/test_integrators.py", cwd=work,
   src=work / "src", tail=17)

The exceptions a function raises belong to its interface. At `lecture-15`, [`src/attractor/integrators.py`](https://github.com/borisbolliet/attractor/blob/lecture-15/src/attractor/integrators.py)
checks the step with `check_step`. `Euler` and `RK4` call it in `__init__`, and `integrate` before its loop:

```python
def check_step(h, t0=0.0, t1=None):
    """Return the number of steps of size h from t0 to t1.

    Raises
    ------
    ValueError
        If h is not a positive number, or if t1 - t0 is not a whole number of
        steps.

    Examples
    --------
    >>> check_step(0.1, 0.0, 1.0)
    10
    >>> check_step(0.3, 0.0, 1.0)
    Traceback (most recent call last):
    ...
    ValueError: (t1 - t0) / h = 3.33333 is not a whole number of steps
    """
    if not h > 0:  # also rejects nan
        raise ValueError(f"step size h must be positive, got {h}")
    if t1 is None:
        return None
    n = round((t1 - t0) / h)
    if not np.isclose(n * h, t1 - t0, rtol=1e-9, atol=0.0):
        raise ValueError(f"(t1 - t0) / h = {(t1 - t0) / h:g} is not a whole number of steps")
    return n
```

We call it with an invalid step in a new interpreter, which imports the `lecture-15` code of `work`:

In [ ]:
py("""
from attractor.integrators import RK4, integrate, rk4_step

for call in (lambda: RK4(-0.1), lambda: integrate(rk4_step, lambda t, y: -y, [1.0], 0.0, 1.0, 0.3)):
    try:
        call()
    except ValueError as e:
        print("ValueError:", e)
""", cwd=work, src=work / "src")

## 5 Mutation testing

A **mutant** is the code with one small change. The tests **kill** it if one of them fails. This mutant
evaluates `k2` at the start of the step:

```diff
- k2 = rhs(t + h / 2, y + h / 2 * k1)
+ k2 = rhs(t, y + h / 2 * k1)
```

Only an equation that depends on $t$ can tell it apart. The cell makes the change in `work`, runs the
tests and restores the file.

In [ ]:
integ = work / "src/attractor/integrators.py"
original = integ.read_text()
edit(integ, "k2 = rhs(t + h / 2, y + h / 2 * k1)", "k2 = rhs(t, y + h / 2 * k1)")
sh(f"{sys.executable} -m pytest -q -p no:cacheprovider", cwd=work, src=work / "src", tail=3)
_ = integ.write_text(original)

## 6 Docstrings and doctests

The docstrings of `lecture-15` follow the NumPy convention. The lines that start with `>>>` in the
section `Examples` are **doctests**: `--doctest-modules` runs them and compares the printed text.
The docstring of `rk4_step`:

```python
def rk4_step(rhs, t, y, h):
    """One classical Runge-Kutta step, a method of order four.

    Parameters
    ----------
    rhs : callable
        Right-hand side ``rhs(t, y)`` of the equation dy/dt = rhs(t, y).
    t : float
        Time at the start of the step.
    y : float or numpy.ndarray
        State at time ``t``.
    h : float
        Step size.

    Returns
    -------
    float or numpy.ndarray
        Approximate state at time ``t + h``.

    Examples
    --------
    One step of dy/dt = -y from y = 1, against the exact value exp(-0.1):

    >>> import math
    >>> y = rk4_step(lambda t, y: -y, 0.0, 1.0, 0.1)
    >>> print(f"{y:.8f}  {math.exp(-0.1):.8f}")
    0.90483750  0.90483742
    """
```

In [ ]:
sh(f"{sys.executable} -m pytest -v -p no:cacheprovider src", cwd=work, src=work / "src", tail=4)

With `k3` from `k1` again, the doctest of `rk4_step` fails and prints both texts:

```diff
- k3 = rhs(t + h / 2, y + h / 2 * k2)
+ k3 = rhs(t + h / 2, y + h / 2 * k1)
```

In [ ]:
edit(integ, "k3 = rhs(t + h / 2, y + h / 2 * k2)", "k3 = rhs(t + h / 2, y + h / 2 * k1)")
sh(f"{sys.executable} -m pytest -q -p no:cacheprovider src", cwd=work, src=work / "src", tail=12)
_ = integ.write_text(original)

Zensical builds the documentation site. [`zensical.toml`](https://github.com/borisbolliet/attractor/blob/lecture-15/zensical.toml) names the pages and the folder of the package:

```toml
# Zensical configuration: the documentation site, built from docs/ and the docstrings.
[project]
site_name = "attractor"
site_url = "https://borisbolliet.github.io/attractor/"
repo_url = "https://github.com/borisbolliet/attractor"
nav = [{ "Home" = "index.md" }, { "API reference" = "api.md" }]

# mkdocstrings reads the docstrings of the package in src/
[project.plugins.mkdocstrings.handlers.python]
paths = ["src"]
options = { docstring_style = "numpy" }
```

Each line `:::` of [`docs/api.md`](https://github.com/borisbolliet/attractor/blob/lecture-15/docs/api.md) inserts the docstrings of a module, read by mkdocstrings:

```markdown
# API reference

::: attractor.integrators

::: attractor.models

::: attractor.trajectory

::: attractor.cycles
```

Zensical is in the extra `docs`. The site is at <https://borisbolliet.github.io/attractor/>.

In [ ]:
if importlib.util.find_spec("zensical") and importlib.util.find_spec("mkdocstrings_handlers"):
    sh(f"{sys.executable} -m zensical build --strict", cwd=work, tail=2)
    print(sorted(str(p.relative_to(work / "site")) for p in (work / "site").rglob("index.html")))
else:
    print('Zensical is not installed in this kernel: python -m pip install -e ".[docs]"')

## 7 Continuous integration

The workflow [`.github/workflows/ci.yml`](https://github.com/borisbolliet/attractor/blob/lecture-15/.github/workflows/ci.yml) runs the job `test` once per combination of the matrix,
then `docs`, which `needs` every run of `test` to succeed:

```yaml
name: tests

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python: ["3.13", "3.14"]
        extras: ["test", "test,fit"]
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: ${{ matrix.python }}
      - run: python -m pip install -e ".[${{ matrix.extras }}]"
      - run: python -m pytest -v --cov=attractor --cov-branch --cov-report=term-missing

  docs:
    needs: test
    if: github.event_name == 'push'
    runs-on: ubuntu-latest
    permissions:
      pages: write
      id-token: write
    environment:
      name: github-pages
      url: ${{ steps.deploy.outputs.page_url }}
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: "3.13"
      - run: python -m pip install -e ".[docs]"
      - run: zensical build --strict
      - uses: actions/upload-pages-artifact@v5
        with:
          path: site
      - id: deploy
        uses: actions/deploy-pages@v5
```

In [ ]:
import itertools
matrix = {"python": ["3.13", "3.14"], "extras": ["test", "test,fit"]}
runs = list(itertools.product(*matrix.values()))
for r in runs:
    print(dict(zip(matrix, r)))
print(f"{len(runs)} runs of test, then docs: {len(runs) + 1} jobs for a push to main")

The exercises and their solutions are at the end of the notes.